# Fase 1 – Análisis de catálogo

Fuente: vistas `v_catalogo`, `v_marcas`, `v_pareto` (`sql/catalogo.sql`) sobre la última captura.
Preguntas: concentración de demanda, participación por marca, rangos de precio.

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

con = sqlite3.connect("../data/db/falabella.db")
cat = pd.read_sql_query("SELECT * FROM v_catalogo", con)
marcas = pd.read_sql_query("SELECT * FROM v_marcas", con)
pareto = pd.read_sql_query("SELECT * FROM v_pareto", con)
print(cat["fecha_captura"].iloc[0], cat.shape)
cat.groupby("categoria").size()

2026-10-21 (13164, 18)


categoria
laptops                         1814
smartphones                     1979
televisores                     1954
zapatillas_deportivas_hombre    1889
zapatillas_deportivas_mujer     1804
zapatillas_urbanas_hombre       1811
zapatillas_urbanas_mujer        1913
dtype: int64

## 1. Concentración de demanda

In [2]:
resumen = cat.groupby("categoria").agg(
    productos=("product_id", "size"),
    con_resenas=("n_resenas", lambda s: (s > 0).mean().round(3) * 100),
    resenas_total=("n_resenas", "sum"),
)
p80 = pareto[pareto["pct_resenas_acum"] >= 80].groupby("categoria")["pct_productos_acum"].min()
resumen["pct_productos_para_80pct_resenas"] = p80
resumen

,productos,con_resenas,resenas_total,pct_productos_para_80pct_resenas
categoria,,,,
laptops,1814,10.9,3260,1.9
smartphones,1979,25.5,160067,0.4
televisores,1954,17.9,15334,1.7
zapatillas_deportivas_hombre,1889,44.5,77704,4.3
zapatillas_deportivas_mujer,1804,35.3,41082,3.7
zapatillas_urbanas_hombre,1811,49.6,146606,2.1
zapatillas_urbanas_mujer,1913,40.6,100926,2.0


In [3]:
top10 = pareto[pareto["ranking"] <= 10].groupby("categoria")["n_resenas"].sum()
(top10 / resumen["resenas_total"] * 100).round(1).rename("pct_resenas_top10")

categoria
laptops                         44.8
smartphones                     89.0
televisores                     54.7
zapatillas_deportivas_hombre    30.6
zapatillas_deportivas_mujer     35.8
zapatillas_urbanas_hombre       52.7
zapatillas_urbanas_mujer        46.8
Name: pct_resenas_top10, dtype: float64

In [4]:
pareto.sort_values("n_resenas", ascending=False).head(15)[["categoria", "marca", "nombre", "n_resenas"]]

,categoria,marca,nombre,n_resenas
1814,smartphones,SAMSUNG,Celular Galaxy S25 Ultra 5G 512GB 12GB Negro,29399
1815,smartphones,SAMSUNG,GALAXY S25 ULTRA 512GB 12GB 200MP TITANIUM GRAY,29345
1816,smartphones,SAMSUNG,Galaxy S24 Ultra 256GB Dual Sim Titanium Negro...,22455
1817,smartphones,SAMSUNG,Galaxy S24 Ultra 256GB Dual Sim Titanio Violet...,22415
9440,zapatillas_urbanas_hombre,ADIDAS ORIGINALS,Zapatillas Urbanas Hombre Samba OG,15018
9441,zapatillas_urbanas_hombre,ADIDAS ORIGINALS,Zapatillas Urbanas Unisex Adulto Samba Og,14745
11251,zapatillas_urbanas_mujer,ADIDAS ORIGINALS,Zapatillas Urbanas Unisex Adulto Samba Og,14745
9442,zapatillas_urbanas_hombre,ADIDAS ORIGINALS,Zapatillas Urbanas Hombre Samba OG,14729
1818,smartphones,SAMSUNG,Galaxy S26 Ultra 256GB 12GB Ram Color Azul,13635
1819,smartphones,SAMSUNG,Celular Galaxy S26 Ultra 5G 12GB 512GB Negro,10119


## 2. Participación por marca

In [5]:
top = marcas.sort_values(["categoria", "pct_resenas"], ascending=[True, False]).groupby("categoria").head(5)
top[["categoria", "marca", "productos", "pct_productos", "resenas", "pct_resenas", "precio_promedio", "rating_promedio"]].reset_index(drop=True)

,categoria,marca,productos,pct_productos,resenas,pct_resenas,precio_promedio,rating_promedio
0,laptops,HP,428,23.6,1995,61.2,3756.0,3.94
1,laptops,LENOVO,598,33.0,578,17.7,4356.0,4.23
2,laptops,APPLE,115,6.3,459,14.1,6323.0,4.61
3,laptops,ASUS,418,23.0,163,5.0,5711.0,4.44
4,laptops,ACER,77,4.2,45,1.4,5208.0,3.79
5,smartphones,SAMSUNG,353,17.8,154777,96.7,2237.0,4.21
6,smartphones,APPLE,287,14.5,3108,1.9,4387.0,4.35
7,smartphones,XIAOMI,463,23.4,1124,0.7,1464.0,4.34
8,smartphones,MOTOROLA,124,6.3,521,0.3,1103.0,4.43
9,smartphones,HONOR,343,17.3,134,0.1,1269.0,4.60


In [6]:
hhi = marcas.groupby("categoria").apply(
    lambda g: pd.Series({
        "marcas": len(g),
        "hhi_productos": (g["pct_productos"] ** 2).sum().round(0),
        "hhi_resenas": (g["pct_resenas"] ** 2).sum().round(0),
    }),
    include_groups=False,
)
hhi

,marcas,hhi_productos,hhi_resenas
categoria,,,
laptops,15.0,2263.0,4285.0
smartphones,29.0,1495.0,9355.0
televisores,19.0,1901.0,5308.0
zapatillas_deportivas_hombre,88.0,620.0,6901.0
zapatillas_deportivas_mujer,89.0,576.0,5711.0
zapatillas_urbanas_hombre,92.0,548.0,4613.0
zapatillas_urbanas_mujer,118.0,524.0,3552.0


HHI (índice Herfindahl-Hirschman) sobre participación en %: < 1500 competitivo, 1500–2500 moderado, > 2500 concentrado.

## 3. Rangos de precio

In [7]:
cat.groupby("categoria")["precio_online"].describe(percentiles=[0.25, 0.5, 0.75]).round(0)

,count,mean,std,min,25%,50%,75%,max
categoria,,,,,,,,
laptops,1814.0,4731.0,2897.0,59.0,2890.0,3790.0,6172.0,26189.0
smartphones,1979.0,1891.0,2209.0,25.0,719.0,1259.0,2399.0,70000.0
televisores,1954.0,1885.0,1749.0,220.0,1099.0,1549.0,2099.0,47999.0
zapatillas_deportivas_hombre,1889.0,275.0,182.0,20.0,149.0,213.0,349.0,1749.0
zapatillas_deportivas_mujer,1804.0,255.0,185.0,20.0,139.0,185.0,300.0,1409.0
zapatillas_urbanas_hombre,1811.0,262.0,141.0,20.0,164.0,239.0,321.0,1080.0
zapatillas_urbanas_mujer,1913.0,234.0,139.0,20.0,136.0,200.0,289.0,1300.0


In [8]:
cat.pivot_table(index="categoria", columns="cuartil_precio", values="precio_online", aggfunc="max").round(0)

cuartil_precio,1,2,3,4
categoria,,,,
laptops,2890.0,3790.0,6180.0,26189.0
smartphones,719.0,1259.0,2399.0,70000.0
televisores,1099.0,1549.0,2099.0,47999.0
zapatillas_deportivas_hombre,149.0,213.0,349.0,1749.0
zapatillas_deportivas_mujer,139.0,183.0,300.0,1409.0
zapatillas_urbanas_hombre,164.0,239.0,321.0,1080.0
zapatillas_urbanas_mujer,136.0,200.0,289.0,1300.0


In [9]:
cat.groupby(["categoria", "cuartil_precio"]).agg(
    descuento_prom=("descuento_pct", "mean"),
    con_lista=("precio_lista", lambda s: s.notna().mean() * 100),
    resenas_prom=("n_resenas", "mean"),
).round(1).unstack("cuartil_precio")

descuento_prom                   con_lista                   resenas_prom                   
cuartil_precio                            1     2     3     4         1     2     3     4            1     2     3      4
categoria                                                                                                                
laptops                                29.5  25.4  21.9  21.3      96.3  96.5  94.9  94.0          1.8   1.4   3.6    0.4
smartphones                            29.8  26.3  26.0  23.0      97.0  97.8  97.2  93.7          5.7   2.8  14.7  300.7
televisores                            40.0  38.0  36.5  35.2      99.2  98.4  96.1  95.3          5.6  12.1   8.0    5.6
zapatillas_deportivas_hombre           38.4  29.8  23.0  26.8      91.5  62.5  51.5  37.9          8.6  59.6  76.8   19.6
zapatillas_deportivas_mujer            42.2  33.1  24.8  24.3      95.6  83.8  45.2  32.8          5.2  21.0  40.9   24.0
zapatillas_urbanas_hombre              41.1  28.9  22.6  21.2      85.7  60.9  58.7  24.1          9.8  33.9  74.2  206.2
zapatillas_urbanas_mujer               40.9  31.8  25.3  21.7      78.7  62.8  60.5  30.1          5.9  34.6  69.3  101.4

In [10]:
vend = cat.groupby(["categoria", "tipo_vendedor"]).agg(
    productos=("product_id", "size"),
    resenas=("n_resenas", "sum"),
    precio_mediano=("precio_online", "median"),
    descuento_prom=("descuento_pct", "mean"),
).round(1)
vend["pct_resenas"] = (vend["resenas"] / vend.groupby(level=0)["resenas"].transform("sum") * 100).round(1)
vend

productos  resenas  precio_mediano  descuento_prom  pct_resenas
categoria                    tipo_vendedor                                                                 
laptops                      1P                    65      554          4099.0            34.2         17.0
                             3P                  1749     2706          3789.0            24.3         83.0
smartphones                  1P                    90     5720          1499.0            17.7          3.6
                             3P                  1889   154347          1245.0            26.7         96.4
televisores                  1P                   118     8123          2049.0            29.2         53.0
                             3P                  1836     7211          1499.0            38.0         47.0
zapatillas_deportivas_hombre 1P                   464    29957           229.4            29.1         38.6
                             3P                  1425    47747           208.9            31.5         61.4
zapatillas_deportivas_mujer  1P                   415    23979           265.3            29.7         58.4
                             3P                  1389    17103           160.9            34.3         41.6
zapatillas_urbanas_hombre    1P                   688   117664           269.9            35.6         80.3
                             3P                  1123    28942           223.1            30.1         19.7
zapatillas_urbanas_mujer     1P                   592    74122           259.0            37.8         73.4
                             3P                  1321    26804           179.0            30.8         26.6

In [11]:
cat.groupby("categoria")["es_patrocinado"].agg(patrocinados="sum", pct=lambda s: round(s.mean() * 100, 1))

,patrocinados,pct
categoria,,
laptops,27,1.5
smartphones,26,1.3
televisores,5,0.3
zapatillas_deportivas_hombre,20,1.1
zapatillas_deportivas_mujer,25,1.4
zapatillas_urbanas_hombre,24,1.3
zapatillas_urbanas_mujer,24,1.3


## Hallazgos

**Concentración.** Entre el 0.4% (smartphones) y el 4.3% (zapatillas deportivas hombre) de los productos acumula el 80% de las reseñas. El top 10 de smartphones concentra el 89%. La cola larga es enorme: entre el 50% y el 89% de los productos no tiene ninguna reseña.

**Marcas.** Samsung domina smartphones (97% de reseñas con 18% de productos) y televisores (72%). Adidas/Adidas Originals domina las cuatro categorías de zapatillas (75–86% de reseñas). En laptops, HP lidera reseñas (61%) pero Lenovo tiene más productos. HHI por reseñas > 3500 en todas: mercados concentrados en demanda, aunque fragmentados en oferta (HHI por productos < 2300).

**Precios.** Cuartil 1 recibe el mayor descuento en todas las categorías (30–42%) y el descuento cae con el precio. En zapatillas, el precio de lista desaparece en los cuartiles altos (solo 24–38% lo tiene): los productos premium no se "tachan". Las reseñas se acumulan en cuartiles 3–4 en smartphones y zapatillas urbanas: la demanda visible está en gama alta.

**1P vs 3P.** Falabella/Tottus/Sodimac venden el 3–38% de los productos pero captan 53–80% de reseñas en televisores y zapatillas urbanas. En smartphones y laptops, el marketplace (3P) captura más del 80% de reseñas.

**Salvedades.** (1) Las reseñas se comparten entre listados del mismo modelo (Galaxy S25 Ultra aparece 2 veces con ~29.4k; Samba OG 3 veces con ~14.7k), lo que infla la concentración: deduplicar por modelo en Fase 2. (2) La categoría `laptops` incluye accesorios (mochilas, monitores) que arrastran el precio mínimo y la mediana de reseñas. (3) `n_resenas` NULL se trata como 0.
